This notebook contains the data analysis and performance comparison between different deep learning models for our first-year Master's project, **"SliceScope"**. This project studies SLA-aware performance monitoring as a closed-loop control problem, conducted under the supervision of Professor Qiong Liu.

**Datasets**
The datasets were taken from: https://github.com/teo-tsou/app_aware_5g/tree/master/dataset

**Methodology & References**
The choices of the deep learning algorithms and their different hyperparameters were taken from:

> Theodoros Tsourdinis, Ilias Chatzistefanidis, Nikos Makris, Thanasis Korakis, Navid Nikaein, et al. *Service-Aware Real-Time Slicing for Virtualized beyond 5G Networks.* Computer Networks, 2025, 247, pp.110445. [DOI: 10.1016/j.comnet.2024.110445](https://doi.org/10.1016/j.comnet.2024.110445). HAL: `hal-04945512`.

# Imports

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FormatStrFormatter, LinearLocator, MaxNLocator


from scipy.stats import wasserstein_distance


from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import MinMaxScaler


import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Bidirectional,
    Conv1D,
    Dense,
    Dropout,
    Flatten,
    GRU,
    Input,
    LSTM,
    MaxPooling1D,
    RepeatVector,
)

from tensorflow.keras.metrics import (
    MeanAbsolutePercentageError,
    RootMeanSquaredError,
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam



In [ ]:
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

# Data analysis

## Dataset 3 & 4 (before/after_slice_sip_rtt)

In [ ]:
bs_df = pd.read_csv('../dataset/before_slice_sip_rtt.csv', sep=";")
as_df = pd.read_csv('../dataset/after_slice_sip_rtt.csv', sep=";")

In [ ]:
# Check if data have any null values
print("Missing values per column:")
print(bs_df.isna().sum())

In [ ]:
# Check if data have any null values
print("Missing values per column:")
print(as_df.isna().sum())

In [ ]:
bs_df['jitter'] = bs_df['response_time_ms'].diff().abs()
as_df['jitter'] = as_df['response_time_ms'].diff().abs()

stats = {
    "Metric": ["Mean", "Median", "Std Dev", "Jitter (Avg Diff)", "Minimum", "Maximum"],
    "Before Slicing (ms)": [
        bs_df['response_time_ms'].mean(),
        bs_df['response_time_ms'].median(),
        bs_df['response_time_ms'].std(),
        bs_df['jitter'].mean(),
        bs_df['response_time_ms'].min(),
        bs_df['response_time_ms'].max()
    ],
    "After Slicing (ms)": [
        as_df['response_time_ms'].mean(),
        as_df['response_time_ms'].median(),
        as_df['response_time_ms'].std(),
        as_df['jitter'].mean(),
        as_df['response_time_ms'].min(),
        as_df['response_time_ms'].max()
    ]
}

stats_df = pd.DataFrame(stats)
print(stats_df)

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.5))

data_to_plot = [bs_df['response_time_ms'], as_df['response_time_ms']]
labels = ['Static', 'Dynamic']
colors = ['#7f7f7f', '#1f77b4']

bplot = ax.boxplot(
    data_to_plot, 
    tick_labels=labels, 
    patch_artist=True, 
    showfliers=False
)

for patch, color in zip(bplot['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.yaxis.set_major_locator(LinearLocator(numticks=4))

ax.yaxis.set_major_formatter(FormatStrFormatter('%d'))

ax.set_ylabel("Latency (ms)")
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
#plt.savefig('static_vs_ai_slicing_latencies.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.5))

window_size = 20

bs_rolling = bs_df['jitter'].rolling(window=window_size).mean()
as_rolling = as_df['jitter'].rolling(window=window_size).mean()


ax.plot(bs_rolling, label='Static', color='red', linewidth=1.2)
ax.plot(as_rolling, label='Dynamic', color='green', linewidth=1.2)

ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)

ax.set_xlabel('Packet Index')
ax.set_ylabel('Avg. Jitter (ms)')
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()

#plt.savefig('jitter_trend_comparison.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()

In [ ]:
plt.figure(figsize=(3.5, 2.5))

def get_cdf(data):
    sorted_data = np.sort(data)
    y_vals = np.arange(len(sorted_data)) / float(len(sorted_data) - 1)
    return sorted_data, y_vals

x_bs, y_bs = get_cdf(bs_df['jitter'])
x_as, y_as = get_cdf(as_df['jitter'])

plt.plot(x_bs, y_bs, label='Static', color='red', linewidth=1.5)
plt.plot(x_as, y_as, label='Dynamic', color='green', linewidth=1.5)

plt.legend(
    bbox_to_anchor=(0., 1.02, 1., .102),
    loc='lower left',
    ncol=2,
    mode="expand",
    borderaxespad=0.,
    frameon=False,
    fontsize=8
)

plt.xlabel('Jitter (ms)')
plt.ylabel('Probability')
plt.grid(True, alpha=0.3)

#plt.savefig('cdf_comparison.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()

## Dataset 2 (ue-lte-network-traffic-stats.csv)

In [ ]:
ts_df = pd.read_csv('../dataset/ue-lte-network-traffic-stats.csv')

In [ ]:
column_map = {
    'UE1: web-rtc': ('UE1', 'WebRTC'),
    'UE1: sipp': ('UE1', 'SIPp'),
    'UE1: web-server': ('UE1', 'Web Server'),
    'UE2: web-rtc': ('UE2', 'WebRTC'),
    'UE2: sipp': ('UE2', 'SIPp'),
    'UE2: web-server': ('UE2', 'Web Server'),
    'UE3: web-rtc': ('UE3', 'WebRTC'),
    'UE3: sipp': ('UE3', 'SIPp'),
    'UE3: web-server': ('UE3', 'Web Server')
}


ue_usage_mbits = {ue: {'WebRTC': 0, 'SIPp': 0, 'Web Server': 0} for ue in ['UE1', 'UE2', 'UE3']}
ue_usage_counts = {ue: {'WebRTC': 0, 'SIPp': 0, 'Web Server': 0} for ue in ['UE1', 'UE2', 'UE3']}


for col, (ue, app) in column_map.items():
    if col in ts_df.columns:
        col_data = ts_df[col]
        
        total_mbits = (col_data.sum() * 8) / 1_000_000
        ue_usage_mbits[ue][app] = total_mbits

        
        active_count = (col_data > 0).sum()
        ue_usage_counts[ue][app] = active_count

# Plot 1
fig1, ax1 = plt.subplots(figsize=(3.5, 2.5))
usage_df_mbits = pd.DataFrame(ue_usage_mbits).T

usage_df_mbits.plot(
    kind='bar', 
    width=0.8, 
    color=['#1f77b4', '#ff7f0e', '#2ca02c'], 
    ax=ax1,
    rot=0
)

ax1.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
ax1.set_xlabel('User Equipment (UE)')
ax1.set_ylabel('Total Volume (Mb)')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

fig1.tight_layout()
#fig1.savefig('ue_data_consumption_grouped.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()

# Plot 2
fig2, ax2 = plt.subplots(figsize=(3.5, 2.5))
usage_df_counts = pd.DataFrame(ue_usage_counts).T

usage_df_counts.plot(
    kind='bar', 
    width=0.8, 
    color=['#1f77b4', '#ff7f0e', '#2ca02c'], 
    ax=ax2,
    rot=0
)

ax2.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)
ax2.set_xlabel('User Equipment (UE)')
ax2.set_ylabel('Active Lines')
ax2.grid(axis='y', linestyle='--', alpha=0.5)

fig2.tight_layout()
#fig2.savefig('ue_usage_frequency_grouped.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()

In [ ]:
# Jitter
fig, axes = plt.subplots(1, 3, figsize=(7.16, 3.5), sharey=True)

ues = ['UE1', 'UE2', 'UE3']
fixed_app_order = ['WebRTC', 'SIPp', 'Web Server']

for i, ue in enumerate(ues):
    
    def get_app_usage(row):
        if row[f'{ue}: web-rtc'] > 0: return 'WebRTC'
        elif row[f'{ue}: sipp'] > 0: return 'SIPp'
        elif row[f'{ue}: web-server'] > 0: return 'Web Server'
        return 'Idle'
            
    cols_to_use = [f'{ue}: web-rtc', f'{ue}: sipp', f'{ue}: web-server', f'{ue}-Jitter']
    df_ue = ts_df[cols_to_use].copy()

    df_ue['App_Usage'] = df_ue.apply(get_app_usage, axis=1)
    
    active_mask = (df_ue[f'{ue}-Jitter'] > 0) & (df_ue['App_Usage'] != 'Idle')
    df_active = df_ue[active_mask].copy()
    
    df_active['Jitter_ms'] = df_active[f'{ue}-Jitter'] * 1000
    
    present_apps = df_active['App_Usage'].unique()
    ordered_apps = [app for app in fixed_app_order if app in present_apps]
    plot_data = [df_active[df_active['App_Usage'] == app]['Jitter_ms'] for app in ordered_apps]
    

    axes[i].boxplot(
        plot_data, 
        tick_labels=ordered_apps,
        showfliers=False, 
        patch_artist=True,
        boxprops=dict(facecolor='white', color='black'),
        medianprops=dict(color='red')
    )
    
    axes[i].set_title(f'{ue}', fontsize=10)
    axes[i].grid(True, alpha=0.3, linestyle='--')
        
axes[0].set_ylabel('Jitter (ms)')

plt.tight_layout()


#plt.savefig('jitter_distribution.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.16, 3.5), sharey=True)

ues = ['UE1', 'UE2', 'UE3']
fixed_app_order = ['WebRTC', 'SIPp', 'Web Server']

palette_map = dict(zip(fixed_app_order, sns.color_palette("viridis", 3)))

for i, ue in enumerate(ues):
    def get_app_usage(row):
        if row[f'{ue}: web-rtc'] > 0: return 'WebRTC'
        elif row[f'{ue}: sipp'] > 0: return 'SIPp'
        elif row[f'{ue}: web-server'] > 0: return 'Web Server'
        return 'Idle'
    
    cols = [f'{ue}: web-rtc', f'{ue}: sipp', f'{ue}: web-server', f'{ue}-CQI']
    df_ue = ts_df[cols].copy()
    df_ue['App_Usage'] = df_ue.apply(get_app_usage, axis=1)
    
    active_mask = df_ue['App_Usage'] != 'Idle'
    df_active = df_ue[active_mask].copy()

    sns.boxenplot(
        data=df_active, 
        x='App_Usage', 
        y=f'{ue}-CQI',
        hue='App_Usage',
        legend=False,
        order=[app for app in fixed_app_order if app in df_active['App_Usage'].unique()],
        ax=axes[i],
        palette=palette_map,
        k_depth="trustworthy"
    )
    
    axes[i].set_xlabel(ue, fontsize=10, fontweight='normal', labelpad=10) 
    
    axes[i].tick_params(axis='x', labelsize=9)
    
    axes[i].set_ylabel('')
    axes[i].set_ylim(0, 16)
    axes[i].grid(True, axis='y', linestyle='--', alpha=0.5)


fig.supylabel('CQI (0-15)', fontsize=10)

plt.tight_layout()

#plt.savefig('cqi_distribution.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.5))
ue_dataframes = []

for ue in ['UE1', 'UE2', 'UE3']:
    # Extract
    temp_df = ts_df[[f'{ue}-CQI', f'{ue}-Jitter']].copy()
    temp_df.columns = ['CQI', 'Jitter']
    
    # Filter and Scale
    temp_df = temp_df[temp_df['Jitter'] > 0].copy()
    temp_df['Jitter'] = temp_df['Jitter'] * 1000 
    
    # Store
    ue_dataframes.append(temp_df)

# Combine all the individual UE dataframes into one
final_df = pd.concat(ue_dataframes)

# Plot
sns.lineplot(
    data=final_df,
    x='CQI',
    y='Jitter',
    marker='o',
    color='crimson',
    linewidth=1.5,
    errorbar='ci',
    ax=ax,
    label='Trend Line'
)

ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=1, frameon=False)

ax.set_ylabel("Avg. Jitter (ms)")
ax.set_xlabel("CQI (Signal Quality)")
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()

#plt.savefig('cqi_vs_jitter_trend.pdf', bbox_inches='tight', pad_inches=0.05)
plt.show()

# ML

## Data prep

In [ ]:
ts_df = pd.read_csv('../dataset/ue-lte-network-traffic-stats.csv')

In [ ]:
# Check if data have any null values
print("Missing values per column:")
print(ts_df.isna().sum())

In [ ]:
key_cols = [
    'UE1: web-rtc', 'UE1: sipp', 'UE1: web-server',
    'UE2: web-rtc', 'UE2: sipp', 'UE2: web-server',
    'UE3: web-rtc', 'UE3: sipp', 'UE3: web-server',
    'UE1-Jitter', 'UE2-Jitter', 'UE3-Jitter',
    'UE1-CQI', 'UE2-CQI', 'UE3-CQI'
]

# Scale the data
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(ts_df[key_cols])
df_normalized = pd.DataFrame(scaled_data, columns=key_cols)


chunk_size = 10000
step_size = 1000  
best_distance = float('inf')
best_start_idx = 0

print("Scanning dataset for the best 10K chunk...")
for start_idx in range(0, len(df_normalized) - chunk_size + 1, step_size):
    end_idx = start_idx + chunk_size
    chunk = df_normalized.iloc[start_idx:end_idx]
    
    chunk_distance = 0
    valid_cols = 0
    
    for col in key_cols:
        # Calculate distance, ignoring NaN (not necessary in our dataset)
        dist = wasserstein_distance(df_normalized[col].dropna(), chunk[col].dropna())
        if not np.isnan(dist):
            chunk_distance += dist
            valid_cols += 1
            
    # Calculate average distance across valid columns
    if valid_cols > 0:
        avg_distance = chunk_distance / valid_cols
        
        if avg_distance < best_distance:
            best_distance = avg_distance
            best_start_idx = start_idx

print("--- Results ---")
print(f"Best 10K chunk starts at index: {best_start_idx}")
print(f"Ending index: {best_start_idx + chunk_size}")
print(f"Average Wasserstein Distance score: {best_distance:.4f}")

# Extract the final, un-normalized 10K chunk
partial_df = ts_df.iloc[best_start_idx:best_start_idx + chunk_size].copy()
#partial_df

In [ ]:
# Use 10k lines of dataset (partial_df)

window_size = 30
# Splits (70, 15,15)
features = [
    'UE1: web-rtc', 'UE1: sipp', 'UE1: web-server',
    'UE2: web-rtc', 'UE2: sipp', 'UE2: web-server',
    'UE3: web-rtc', 'UE3: sipp', 'UE3: web-server',
    'UE1-Jitter', 'UE2-Jitter', 'UE3-Jitter',
    'UE1-CQI', 'UE2-CQI', 'UE3-CQI'
]

total_rows = len(partial_df) # 10,000 rows
train_cutoff = int(total_rows * 0.70) # Row 7,000
val_cutoff = train_cutoff + int(total_rows * 0.15) # Row 8,500

# Chronological Split (Sequential slicing)
train_df = partial_df.iloc[:train_cutoff].copy()           # Rows 0 to 6,999
val_df = partial_df.iloc[train_cutoff:val_cutoff].copy()   # Rows 7,000 to 8,499
test_df = partial_df.iloc[val_cutoff:].copy()              # Rows 8,500 to 10,000

print(f"Training set: {len(train_df)} rows")
print(f"Validation set: {len(val_df)} rows")
print(f"Testing set: {len(test_df)} rows")


# Scale the data
scaler = MinMaxScaler()

# fit the scaler on the Training data
train_scaled = scaler.fit_transform(train_df[features].fillna(0))

# Transform Validation and Test sets using the scaling rules learned from Training
val_scaled = scaler.transform(val_df[features].fillna(0))
test_scaled = scaler.transform(test_df[features].fillna(0))

# Pad the arrays(val & test)
# val
val_scaled = np.vstack((train_scaled[-window_size:], val_scaled))
# test
test_scaled = np.vstack((val_scaled[-window_size:], test_scaled))

In [ ]:
# Use the full dataset (ts_df)

# window_size = 30

# features = [
#     'UE1: web-rtc', 'UE1: sipp', 'UE1: web-server',
#     'UE2: web-rtc', 'UE2: sipp', 'UE2: web-server',
#     'UE3: web-rtc', 'UE3: sipp', 'UE3: web-server',
#     'UE1-Jitter', 'UE2-Jitter', 'UE3-Jitter',
#     'UE1-CQI', 'UE2-CQI', 'UE3-CQI'
# ]

# total_rows = len(ts_df)
# train_cutoff = int(total_rows * 0.70)
# val_cutoff = train_cutoff + int(total_rows * 0.15)

# # Chronological Split (Sequential slicing)
# train_df = ts_df.iloc[:train_cutoff].copy()
# val_df = ts_df.iloc[train_cutoff:val_cutoff].copy()
# test_df = ts_df.iloc[val_cutoff:].copy()

# print(f"Training set: {len(train_df)} rows")
# print(f"Validation set: {len(val_df)} rows")
# print(f"Testing set: {len(test_df)} rows")



# scaler = MinMaxScaler()


# train_scaled = scaler.fit_transform(train_df[features].fillna(0))


# val_scaled = scaler.transform(val_df[features].fillna(0))
# test_scaled = scaler.transform(test_df[features].fillna(0))


# val_scaled = np.vstack((train_scaled[-window_size:], val_scaled))
# test_scaled = np.vstack((val_scaled[-window_size:], test_scaled))

## Helper functions

In [ ]:
def create_sliding_window(data, window_size=30, horizon=1, direct=1):
    X, y = [], []
    
    for i in range(len(data) - window_size - horizon + 1):
        # Extract the sequence of past data (30 rows)
        X.append(data[i : (i + window_size)])
        
        if direct == 1:
            # Grab the single target value exactly 'horizon' steps ahead
            y.append(data[i + window_size + horizon - 1])
            
        elif direct == 0:
            # Grab all steps from t+1 up to t+h
            future_slice = data[i + window_size : i + window_size + horizon]
            # Calculate the average across those time steps for each feature
            y.append(np.mean(future_slice, axis=0))
            
    return np.array(X), np.array(y)


# window_size = 30
# horizon_5 = 5

# Direct Data (Predicting exactly t+5)
# X_train_dir, y_train_dir = create_sliding_window(train_scaled, window_size, horizon_5, 1)

# Aggregated Data (Predicting the average of t+1 through t+5)
# X_train_agg, y_train_agg = create_sliding_window(train_scaled, window_size, horizon_5, 0)

# Recursive Data (Predicting t+1, to be used in a loop later)
# X_train_rec, y_train_rec = create_sliding_window(train_scaled, window_size, 1, 1)

In [ ]:
# Custom Evaluation Metric
def tolerance_accuracy(y_true, y_pred, tol=0.05):
    """
    Calculates the percentage of predictions that fall within a (tol) tolerance 
    of the actual values.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    
    epsilon = K.epsilon() 
    
    absolute_error = tf.abs(y_pred - y_true)
    relative_error = absolute_error / (tf.abs(y_true) + epsilon)
    
    within_tolerance = tf.less(relative_error, tol)
    
    accuracy = tf.reduce_mean(tf.cast(within_tolerance, tf.float32))
    
    return accuracy

In [ ]:
# Max epochs from the paper's specifications
EPOCH_LIMITS = {
    'fnn': 568,
    'cnn': 264,
    'lstm': 97,
    'gru': 61,
    'bi-lstm': 56,
    'cnn-lstm': 24
}

def train_optimized_model(model, model_name, X_train, y_train, X_val, y_val, batch_size=64):
    """
    Trains a model using tf.data pipelines, dynamic learning rates, and early stopping.
    Enforces the maximum epochs defined in the reference paper.
    """
    max_epochs = EPOCH_LIMITS.get(model_name.lower(), 100)
    
    # Calculate a dynamic patience scale (20% of max epochs, minimum of 3)
    patience_scale = max(3, max_epochs // 20)

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_dataset = train_dataset.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_dataset = val_dataset.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Configure Callbacks
    early_stop = EarlyStopping(
        monitor='val_loss', 
        patience=patience_scale,       # Wait dynamically based on model
        restore_best_weights=True,     # Revert to the best epoch automatically
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5,                    # Cut learning rate in half if plateaued
        patience=patience_scale // 2,  # Trigger LR reduction before Early Stopping kicks in
        min_lr=1e-6,
        verbose=1
    )

    # Train the Model using the pipeline
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=max_epochs,
        callbacks=[early_stop, reduce_lr],
        verbose=2 # Cleaner console output for hundreds of epochs
    )

    return history

In [ ]:
def train_model_wrapper(model, model_name, X_train, y_train, X_val, y_val, epochs, batch_size=64, optimized=True):
    """
    Toggles between standard Keras .fit() and the optimized tf.data pipeline.
    """
    if optimized:
        print(f"--- Training {model_name.upper()} (Optimized Pipeline) ---")
        history = train_optimized_model(
            model=model, 
            model_name=model_name, 
            X_train=X_train, 
            y_train=y_train, 
            X_val=X_val, 
            y_val=y_val, 
            batch_size=batch_size
        )
    else:
        print(f"--- Training {model_name.upper()} (Standard .fit) ---")
        history = model.fit(
            X_train, y_train, 
            validation_data=(X_val, y_val), 
            epochs=epochs, 
            batch_size=batch_size, 
            verbose=1
        )
    return history

In [ ]:
def plot_training_history(history, model_name):
    """
    Plots the training and validation Loss and Tolerance Accuracy in separate figures
    """
    
    # Plot Loss (MSE)
    fig1, ax1 = plt.subplots(figsize=(3.5, 2.5))

    ax1.plot(history.history['loss'], label='Train Loss', color='blue', linewidth=1.2)
    ax1.plot(history.history['val_loss'], label='Val Loss', color='red', linestyle='--', linewidth=1.2)
    
    ax1.set_xlabel('Epochs', fontsize=8)
    ax1.set_ylabel('Loss (MSE)', fontsize=8)
    ax1.tick_params(axis='both', labelsize=8)
    ax1.legend(fontsize=8, frameon=False) 
    ax1.grid(True, linestyle=':', alpha=0.5, color='gray')

    plt.tight_layout()
    plt.savefig(f'Loss_{model_name.replace("-", "_")}.pdf', format='pdf', bbox_inches='tight')
    plt.show()

    
    # Plot Custom Metric (Tolerance Accuracy)
    fig2, ax2 = plt.subplots(figsize=(3.5, 2.5))

    ax2.plot(history.history['tolerance_accuracy'], label='Train Acc', color='blue', linewidth=1.2)
    ax2.plot(history.history['val_tolerance_accuracy'], label='Val Acc', color='red', linestyle='--', linewidth=1.2)
    
    ax2.set_xlabel('Epochs', fontsize=8)
    ax2.set_ylabel('Tol. Accuracy', fontsize=8)
    ax2.tick_params(axis='both', labelsize=8)
    ax2.legend(fontsize=8, frameon=False)
    ax2.grid(True, linestyle=':', alpha=0.5, color='gray')

    plt.tight_layout()
    plt.savefig(f'Accuracy_{model_name.replace("-", "_")}.pdf', format='pdf', bbox_inches='tight')
    plt.show()

In [ ]:
def plot_actual_pred(actual, predicted, y_label, x_steps, feature_name, model_type, horizon):
    """
    Plots Actual vs Predicted data
    Dynamically handles labels and filenames for Direct, Recursive, and Aggregated models
    """
    # ==========================================
    # FOR CLEANER PLOTS
    # Scale down by 1000 (e.g., 60000 -> 60)
    # actual = actual / 1000
    # predicted = predicted / 1000
    # ==========================================
    
    fig, ax = plt.subplots(figsize=(3.5, 2.5))

    # Determine legend labels based on the model strategy
    model_type = model_type.lower()
    if model_type == 'direct':
        actual_label = 'Actual'
        pred_label = f'Direct Pred (t+{horizon})'
    elif model_type == 'recursive':
        actual_label = 'Actual'
        pred_label = f'Rec Pred (t+{horizon})'
    elif model_type == 'aggregated':
        actual_label = 'Actual Average'
        pred_label = f'Agg Pred (h={horizon})'
    else:
        actual_label = 'Actual'
        pred_label = 'Predicted'

    ax.plot(actual, label=actual_label, color='blue', linestyle='-', linewidth=1.2, alpha=0.8)
    ax.plot(predicted, label=pred_label, color='red', linestyle='--', linewidth=1.2)

    ax.set_xlabel(f'Time Slots (First {x_steps})', fontsize=8)
    ax.set_ylabel(y_label, fontsize=8)
    ax.tick_params(axis='both', which='major', labelsize=8)

    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, 
              fontsize=7, framealpha=1.0, edgecolor='black')

    ax.grid(True, linestyle=':', alpha=0.5, color='gray')
    
    ax.ticklabel_format(style='plain', axis='y', useOffset=False)
    plt.tight_layout()

    safe_name = feature_name.replace(":", "").replace(" ", "_")
    plt.savefig(f'{model_type.capitalize()}_h{horizon}_{safe_name}.pdf', format='pdf', bbox_inches='tight')
    
    plt.show()

In [ ]:
def generate_recursive_predictions(model, X_test, target_horizon, steps_to_predict):
    """
    Steps forward 'target_horizon' times for each window in X_test 
    to generate recursive predictions.
    """
    recursive_predictions = []
    
    for i in range(steps_to_predict):
        current_window = X_test[i:i+1] 
        
        for step in range(target_horizon):
            # Predict t+1
            pred = model(current_window, training=False)
            
            # Save the prediction if we hit our target horizon
            if step == target_horizon - 1:
                recursive_predictions.append(pred[0])
                
            # Reshape and append to the window, dropping the oldest step
            pred_reshaped = np.reshape(pred, (1, 1, pred.shape[-1]))
            current_window = np.append(current_window[:, 1:, :], pred_reshaped, axis=1)
            
    return np.array(recursive_predictions)

## FNN

In [ ]:
def build_fnn_model(input_shape, output_dim):
    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(Flatten())
    
    # First layer (25 units)
    model.add(Dense(25, activation='relu'))

    # Second layer (25 units)
    model.add(Dense(25, activation='relu'))

    # output layer
    model.add(Dense(output_dim, activation='linear'))
    
    model.compile(
            optimizer='adam', 
            loss='mse', 
            metrics=[
                'mae', 
                RootMeanSquaredError(name='rmse'), 
                MeanAbsolutePercentageError(name='mape'),
                tolerance_accuracy
            ]
    )
    return model

### Direct

In [ ]:
horizon_d = 1

X_train_dir, y_train_dir = create_sliding_window(train_scaled, window_size, horizon_d, 1)

X_val_dir, y_val_dir = create_sliding_window(val_scaled, window_size, horizon_d, 1)

X_test_dir, y_test_dir = create_sliding_window(test_scaled, window_size, horizon_d, 1)

print(np.shape(X_train_dir))
print(np.shape(y_train_dir))

print()

print(np.shape(X_val_dir))
print(np.shape(y_val_dir))

print()

print(np.shape(X_test_dir))
print(np.shape(y_test_dir))

In [ ]:
# build and summary
fnn_model_dir = build_fnn_model(
    input_shape=(X_train_dir.shape[1], X_train_dir.shape[2]), 
    output_dim=y_train_dir.shape[1]
)
fnn_model_dir.summary()

In [ ]:
# Train the model
history_fnn_dir = train_model_wrapper(fnn_model_dir, 'fnn', X_train_dir, y_train_dir, X_val_dir, y_val_dir, epochs=568, optimized=True)

In [ ]:
plot_training_history(history_fnn_dir, "FNN_Model_direct")

In [ ]:
# Save the model
fnn_model_dir.save(f"fnn_direct_h_{horizon_d}.keras")

In [ ]:
# Load the model
loaded_dir_model = tf.keras.models.load_model(
    f"fnn_direct_h_{horizon_d}.keras", 
    compile=False
)

In [ ]:
# ==========================================
# PLOT CONFIGURATION
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
# ==========================================

print("=== Generating FNN Direct Prediction for Plotting ===")

# Generate the scaled predictions
y_pred_scaled = loaded_dir_model.predict(X_test_dir, verbose=0)

# INVERSE TRANSFORM to get original values back
y_test_unscaled = scaler.inverse_transform(y_test_dir)
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

# Extract actual and predicted arrays, strictly sliced
actual = y_test_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

# Plot using our unified function
plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="direct", 
    horizon=horizon_d
)

### Recursive

In [ ]:
# Data
horizon_r = 1

X_train_rec, y_train_rec = create_sliding_window(train_scaled, window_size, horizon_r, 1)
X_val_rec, y_val_rec = create_sliding_window(val_scaled, window_size, horizon_r, 1)
X_test_rec, y_test_rec = create_sliding_window(test_scaled, window_size, horizon_r, 1)

print("Recursive Data Shapes:")
print("Train X:", np.shape(X_train_rec), "| Train y:", np.shape(y_train_rec))
print("Val X:  ", np.shape(X_val_rec), "| Val y:  ", np.shape(y_val_rec))
print("Test X: ", np.shape(X_test_rec), "| Test y: ", np.shape(y_test_rec))

In [ ]:
# build and summary
fnn_model_rec = build_fnn_model(
    input_shape=(X_train_rec.shape[1], X_train_rec.shape[2]), 
    output_dim=y_train_rec.shape[1]
)
fnn_model_rec.summary()

In [ ]:
# Train the model
history_fnn_rec = train_model_wrapper(fnn_model_rec, 'fnn', X_train_rec, y_train_rec, X_val_rec, y_val_rec, epochs=568, optimized=True)

In [ ]:
plot_training_history(history_fnn_rec, "FNN_Model_recursive")

In [ ]:
# Save the model
fnn_model_rec.save(f"fnn_recursive_h_{horizon_r}.keras")

In [ ]:
# Load the model
loaded_rec_model = tf.keras.models.load_model(
    f"fnn_recursive_h_{horizon_r}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
TARGET_HORIZON = 5              # Steps forward
# ==========================================
print(f"=== Generating FNN Recursive Predictions for t+{TARGET_HORIZON} ===")

valid_length = len(X_test_rec) - TARGET_HORIZON + 1
steps_to_predict = min(X_STEPS_TO_PLOT, valid_length)

y_pred_rec_scaled = generate_recursive_predictions(loaded_rec_model, X_test_rec, TARGET_HORIZON, steps_to_predict)

_, y_test_aligned_scaled = create_sliding_window(test_scaled, window_size, TARGET_HORIZON, direct=1)
y_test_aligned_scaled = y_test_aligned_scaled[:valid_length]

y_pred_unscaled = scaler.inverse_transform(y_pred_rec_scaled)
y_actual_unscaled = scaler.inverse_transform(y_test_aligned_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(actual, predicted, Y_LABEL, X_STEPS_TO_PLOT, FEATURE_NAME, "recursive", TARGET_HORIZON)

### Aggregated

In [ ]:
# Data
horizon_a = 5

X_train_agg, y_train_agg = create_sliding_window(train_scaled, window_size, horizon_a, 0)
X_val_agg, y_val_agg = create_sliding_window(val_scaled, window_size, horizon_a, 0)
X_test_agg, y_test_agg = create_sliding_window(test_scaled, window_size, horizon_a, 0)

print("Aggregated Data Shapes:")
print("Train X:", np.shape(X_train_agg), "| Train y:", np.shape(y_train_agg))
print("Val X:  ", np.shape(X_val_agg), "| Val y:  ", np.shape(y_val_agg))
print("Test X: ", np.shape(X_test_agg), "| Test y: ", np.shape(y_test_agg))

In [ ]:
# build and summary
fnn_model_agg = build_fnn_model(
    input_shape=(X_train_agg.shape[1], X_train_agg.shape[2]), 
    output_dim=y_train_agg.shape[1]
)
fnn_model_agg.summary()

In [ ]:
# Train the model
history_fnn_agg = train_model_wrapper(fnn_model_agg, 'fnn', X_train_agg, y_train_agg, X_val_agg, y_val_agg, epochs=568, optimized=True)

In [ ]:
plot_training_history(history_fnn_agg, "FNN_Model_aggregated")

In [ ]:
# Save the model
fnn_model_agg.save(f"fnn_aggregated_h_{horizon_a}.keras")

In [ ]:
# Load the model
loaded_agg_model = tf.keras.models.load_model(
    f"fnn_aggregated_h_{horizon_a}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
horizon_a = 5                   
# ==========================================

print(f"=== Generating FNN Aggregated Prediction for Average of t+1 to t+{horizon_a} ===")

y_pred_scaled = loaded_agg_model.predict(X_test_agg, verbose=0)

y_actual_unscaled = scaler.inverse_transform(y_test_agg) 
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="aggregated", 
    horizon=horizon_a
)

## CNN

In [ ]:
def build_cnn_model(input_shape, output_dim):
    model = Sequential()
    
    # Input layer
    model.add(Input(shape=input_shape))
    
    # 1D Convolutional Layer (64 filters, kernel size 2)
    model.add(Conv1D(filters=64, kernel_size=2, activation='relu'))
    
    # Max Pooling Layer (pool size 2)
    model.add(MaxPooling1D(pool_size=2))
    
    # Flatten to transition from 3D to 2D
    model.add(Flatten())
    
    # Fully Connected Hidden Layer (50 units)
    model.add(Dense(50, activation='relu'))
    
    # Output layer
    model.add(Dense(output_dim, activation='linear'))
    
    model.compile(
        optimizer='adam', 
        loss='mse', 
        metrics=[
            'mae', 
            RootMeanSquaredError(name='rmse'), 
            MeanAbsolutePercentageError(name='mape'),
            tolerance_accuracy
        ]
    )
    
    return model

### Direct

In [ ]:
horizon_d = 1

X_train_dir, y_train_dir = create_sliding_window(train_scaled, window_size, horizon_d, 1)

X_val_dir, y_val_dir = create_sliding_window(val_scaled, window_size, horizon_d, 1)

X_test_dir, y_test_dir = create_sliding_window(test_scaled, window_size, horizon_d, 1)

print(np.shape(X_train_dir))
print(np.shape(y_train_dir))

print()

print(np.shape(X_val_dir))
print(np.shape(y_val_dir))

print()

print(np.shape(X_test_dir))
print(np.shape(y_test_dir))

In [ ]:
# Build and summarize the model
cnn_model_dir = build_cnn_model(
    input_shape=(X_train_dir.shape[1], X_train_dir.shape[2]), 
    output_dim=y_train_dir.shape[1]
)
cnn_model_dir.summary()

In [ ]:
# Train the model
history_cnn_dir = train_model_wrapper(cnn_model_dir, 'cnn', X_train_dir, y_train_dir, X_val_dir, y_val_dir, epochs=264, optimized=True)

In [ ]:
plot_training_history(history_cnn_dir, "CNN_Model_direct")

In [ ]:
# Save the model
cnn_model_dir.save(f"cnn_direct_h_{horizon_d}.keras")

In [ ]:
# Load the model
loaded_cnn_dir_model = tf.keras.models.load_model(
    f"cnn_direct_h_{horizon_d}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
# ==========================================

print("=== Generating CNN Direct Prediction for Plotting ===")

y_pred_scaled = loaded_cnn_dir_model.predict(X_test_dir, verbose=0)


y_test_unscaled = scaler.inverse_transform(y_test_dir)
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_test_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="direct", 
    horizon=horizon_d
)

### Recursive

In [ ]:
# Data
horizon_r = 1

X_train_rec, y_train_rec = create_sliding_window(train_scaled, window_size, horizon_r, 1)
X_val_rec, y_val_rec = create_sliding_window(val_scaled, window_size, horizon_r, 1)
X_test_rec, y_test_rec = create_sliding_window(test_scaled, window_size, horizon_r, 1)

print("Recursive Data Shapes:")
print("Train X:", np.shape(X_train_rec), "| Train y:", np.shape(y_train_rec))
print("Val X:  ", np.shape(X_val_rec), "| Val y:  ", np.shape(y_val_rec))
print("Test X: ", np.shape(X_test_rec), "| Test y: ", np.shape(y_test_rec))

In [ ]:
# Build and summary
cnn_model_rec = build_cnn_model(
    input_shape=(X_train_rec.shape[1], X_train_rec.shape[2]), 
    output_dim=y_train_rec.shape[1]
)
cnn_model_rec.summary()

In [ ]:
# Train the model
history_cnn_rec = train_model_wrapper(cnn_model_rec, 'cnn', X_train_rec, y_train_rec, X_val_rec, y_val_rec, epochs=264, optimized=True)

In [ ]:
plot_training_history(history_cnn_rec, "CNN_Model_recursive")

In [ ]:
# Save the model
cnn_model_rec.save(f"cnn_recursive_h_{horizon_r}.keras")

In [ ]:
# Load the model
loaded_cnn_rec_model = tf.keras.models.load_model(
    f"cnn_recursive_h_{horizon_r}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
TARGET_HORIZON = 5
# ==========================================

print(f"=== Generating CNN Recursive Predictions for t+{TARGET_HORIZON} ===")

valid_length = len(X_test_rec) - TARGET_HORIZON + 1
steps_to_predict = min(X_STEPS_TO_PLOT, valid_length)

y_pred_rec_scaled = generate_recursive_predictions(
    loaded_cnn_rec_model, 
    X_test_rec, 
    TARGET_HORIZON, 
    steps_to_predict
)

_, y_test_aligned_scaled = create_sliding_window(test_scaled, window_size, TARGET_HORIZON, direct=1)
y_test_aligned_scaled = y_test_aligned_scaled[:valid_length]

y_pred_unscaled = scaler.inverse_transform(y_pred_rec_scaled)
y_actual_unscaled = scaler.inverse_transform(y_test_aligned_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="recursive", 
    horizon=TARGET_HORIZON
)

### Aggregated

In [ ]:
# Data
horizon_a = 5

X_train_agg, y_train_agg = create_sliding_window(train_scaled, window_size, horizon_a, 0)
X_val_agg, y_val_agg = create_sliding_window(val_scaled, window_size, horizon_a, 0)
X_test_agg, y_test_agg = create_sliding_window(test_scaled, window_size, horizon_a, 0)

print("Aggregated Data Shapes:")
print("Train X:", np.shape(X_train_agg), "| Train y:", np.shape(y_train_agg))
print("Val X:  ", np.shape(X_val_agg), "| Val y:  ", np.shape(y_val_agg))
print("Test X: ", np.shape(X_test_agg), "| Test y: ", np.shape(y_test_agg))

In [ ]:
# Build and summary
cnn_model_agg = build_cnn_model(
    input_shape=(X_train_agg.shape[1], X_train_agg.shape[2]), 
    output_dim=y_train_agg.shape[1]
)
cnn_model_agg.summary()

In [ ]:
# Train the model
history_cnn_agg = train_model_wrapper(cnn_model_agg, 'cnn', X_train_agg, y_train_agg, X_val_agg, y_val_agg, epochs=264, optimized=True)

In [ ]:
plot_training_history(history_cnn_agg, "CNN_Model_aggregated")

In [ ]:
# Save the model
cnn_model_agg.save(f"cnn_aggregated_h_{horizon_a}.keras")

In [ ]:
# Load the model
loaded_cnn_agg_model = tf.keras.models.load_model(
    f"cnn_aggregated_h_{horizon_a}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
horizon_a = 5                   
# ==========================================

print(f"=== Generating CNN Aggregated Prediction for Average of t+1 to t+{horizon_a} ===")

y_pred_scaled = loaded_cnn_agg_model.predict(X_test_agg, verbose=0)

y_actual_unscaled = scaler.inverse_transform(y_test_agg) 
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="aggregated", 
    horizon=horizon_a
)

## LSTM

In [ ]:
def build_lstm_model(input_shape, output_dim):
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # First layer (25 units)
    model.add(LSTM(25, return_sequences=True))

    # Second layer (25 units)
    model.add(LSTM(25, return_sequences=False))

    # output layer
    model.add(Dense(output_dim, activation='linear'))
    
    model.compile(optimizer='adam', loss='mse', metrics=[
                'mae', 
                RootMeanSquaredError(name='rmse'), 
                MeanAbsolutePercentageError(name='mape'),
                tolerance_accuracy
            ])
    return model

### Direct 

In [ ]:
horizon_d = 1

X_train_dir, y_train_dir = create_sliding_window(train_scaled, window_size, horizon_d, 1)

X_val_dir, y_val_dir = create_sliding_window(val_scaled, window_size, horizon_d, 1)

X_test_dir, y_test_dir = create_sliding_window(test_scaled, window_size, horizon_d, 1)

print(np.shape(X_train_dir))
print(np.shape(y_train_dir))

print()

print(np.shape(X_val_dir))
print(np.shape(y_val_dir))

print()

print(np.shape(X_test_dir))
print(np.shape(y_test_dir))


In [ ]:
lstm_model_dir = build_lstm_model(input_shape=(X_train_dir.shape[1], X_train_dir.shape[2]), output_dim=y_train_dir.shape[1])
lstm_model_dir.summary()

In [ ]:
# Train the model
history_lstm_dir = train_model_wrapper(lstm_model_dir, 'lstm', X_train_dir, y_train_dir, X_val_dir, y_val_dir, epochs=97, optimized=True)

In [ ]:
plot_training_history(history_lstm_dir, "LSTM_Model_direct")

In [ ]:
# Save the model
lstm_model_dir.save(f"lstm_direct_h_{horizon_d}.keras")
print("Direct model successfully saved as 'lstm_direct.keras'")

In [ ]:
# Load the model
loaded_direct_model = tf.keras.models.load_model(
    f"lstm_direct_h_{horizon_d}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               # 0-14
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
# ==========================================

print("=== Generating LSTM Direct Prediction for Plotting ===")

y_pred_scaled = loaded_direct_model.predict(X_test_dir, verbose=0)

y_test_unscaled = scaler.inverse_transform(y_test_dir)
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_test_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="direct", 
    horizon=horizon_d
)

### Recursive

In [ ]:
horizon_r = 1


X_train_rec, y_train_rec = create_sliding_window(train_scaled, window_size, horizon_r, 1)

X_val_rec, y_val_rec = create_sliding_window(val_scaled, window_size, horizon_r, 1)

X_test_rec, y_test_rec = create_sliding_window(test_scaled, window_size, horizon_r, 1)

print("Train X:", np.shape(X_train_rec), "| Train y:", np.shape(y_train_rec))
print("Val X:  ", np.shape(X_val_rec), "| Val y:  ", np.shape(y_val_rec))
print("Test X: ", np.shape(X_test_rec), "| Test y: ", np.shape(y_test_rec))

In [ ]:
lstm_model_rec = build_lstm_model(
    input_shape=(X_train_rec.shape[1], X_train_rec.shape[2]), 
    output_dim=y_train_rec.shape[1]
)
lstm_model_rec.summary()

In [ ]:
# Train the model
history_lstm_rec = train_model_wrapper(lstm_model_rec, 'lstm', X_train_rec, y_train_rec, X_val_rec, y_val_rec, epochs=97, optimized=True)

In [ ]:
plot_training_history(history_lstm_rec, "LSTM_Model_recursive")

In [ ]:
# Save the model
lstm_model_rec.save(f"lstm_recursive_h_{horizon_r}.keras")

In [ ]:
# Load the model
loaded_rec_model = tf.keras.models.load_model(
    f"lstm_recursive_h_{horizon_r}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
TARGET_HORIZON = 5             
# ==========================================
print(f"=== Generating LSTM Recursive Predictions for t+{TARGET_HORIZON} ===")

valid_length = len(X_test_rec) - TARGET_HORIZON + 1
steps_to_predict = min(X_STEPS_TO_PLOT, valid_length)

y_pred_rec_scaled = generate_recursive_predictions(loaded_rec_model, X_test_rec, TARGET_HORIZON, steps_to_predict)

_, y_test_aligned_scaled = create_sliding_window(test_scaled, window_size, TARGET_HORIZON, direct=1)
y_test_aligned_scaled = y_test_aligned_scaled[:valid_length]

y_pred_unscaled = scaler.inverse_transform(y_pred_rec_scaled)
y_actual_unscaled = scaler.inverse_transform(y_test_aligned_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(actual, predicted, Y_LABEL, X_STEPS_TO_PLOT, FEATURE_NAME, "recursive", TARGET_HORIZON)

### Aggregated

In [ ]:
horizon_a = 5 

X_train_agg, y_train_agg = create_sliding_window(train_scaled, window_size, horizon_a, 0)
X_val_agg, y_val_agg = create_sliding_window(val_scaled, window_size, horizon_a, 0)
X_test_agg, y_test_agg = create_sliding_window(test_scaled, window_size, horizon_a, 0)

print("Train X:", np.shape(X_train_agg), "| Train y:", np.shape(y_train_agg))
print("Val X:  ", np.shape(X_val_agg), "| Val y:  ", np.shape(y_val_agg))
print("Test X: ", np.shape(X_test_agg), "| Test y: ", np.shape(y_test_agg))

In [ ]:
lstm_model_agg = build_lstm_model(
    input_shape=(X_train_agg.shape[1], X_train_agg.shape[2]), 
    output_dim=y_train_agg.shape[1]
)
lstm_model_agg.summary()

In [ ]:
# Train the model
history_lstm_agg = train_model_wrapper(lstm_model_agg, 'lstm', X_train_agg, y_train_agg, X_val_agg, y_val_agg, epochs=97, optimized=True)

In [ ]:
plot_training_history(history_lstm_agg, "LSTM_Model_aggregated")

In [ ]:
# Save the model
lstm_model_agg.save(f"lstm_aggregated_h_{horizon_a}.keras")

In [ ]:
# Load the model
loaded_agg_model = tf.keras.models.load_model(
    f"lstm_aggregated_h_{horizon_a}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
horizon_a = 5
# ==========================================

print(f"=== Generating LSTM Aggregated Prediction for Average of t+1 to t+{horizon_a} ===")

y_pred_scaled = loaded_agg_model.predict(X_test_agg, verbose=0)

y_actual_unscaled = scaler.inverse_transform(y_test_agg) 
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="aggregated", 
    horizon=horizon_a
)

## GRU

In [ ]:
# build

def build_gru_model(input_shape, output_dim):
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # First layer (25 units)
    model.add(GRU(25, return_sequences=True))

    # Second layer (25 units)
    model.add(GRU(25, return_sequences=False))

    # Output layer
    model.add(Dense(output_dim, activation='linear'))
    
    model.compile(optimizer='adam', loss='mse', metrics=[
                'mae', 
                RootMeanSquaredError(name='rmse'), 
                MeanAbsolutePercentageError(name='mape'),
                tolerance_accuracy
            ])
    
    return model

### Direct

In [ ]:
# Data
horizon_d = 1

X_train_dir, y_train_dir = create_sliding_window(train_scaled, window_size, horizon_d, 1)
X_val_dir, y_val_dir = create_sliding_window(val_scaled, window_size, horizon_d, 1)
X_test_dir, y_test_dir = create_sliding_window(test_scaled, window_size, horizon_d, 1)

print("Direct Data Shapes:")
print("Train X:", np.shape(X_train_dir), "| Train y:", np.shape(y_train_dir))
print("Val X:  ", np.shape(X_val_dir), "| Val y:  ", np.shape(y_val_dir))
print("Test X: ", np.shape(X_test_dir), "| Test y: ", np.shape(y_test_dir))

In [ ]:
# build and summary
gru_model_dir = build_gru_model(
    input_shape=(X_train_dir.shape[1], X_train_dir.shape[2]), 
    output_dim=y_train_dir.shape[1]
)
gru_model_dir.summary()

In [ ]:
# Train the model
history_gru_dir = train_model_wrapper(gru_model_dir, 'gru', X_train_dir, y_train_dir, X_val_dir, y_val_dir, epochs=61, optimized=True)

In [ ]:
plot_training_history(history_gru_dir, "GRU_Model_direct")

In [ ]:
# Save the model
gru_model_dir.save(f"gru_direct_h_{horizon_d}.keras")

In [ ]:
# Load the model
loaded_dir_model = tf.keras.models.load_model(
    f"gru_direct_h_{horizon_d}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
# ==========================================

print("=== Generating GRU Direct Prediction for Plotting ===")

y_pred_scaled = loaded_dir_model.predict(X_test_dir, verbose=0)

y_test_unscaled = scaler.inverse_transform(y_test_dir)
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_test_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="direct", 
    horizon=horizon_d
)

### Recursive

In [ ]:
# Data
horizon_r = 1

X_train_rec, y_train_rec = create_sliding_window(train_scaled, window_size, horizon_r, 1)
X_val_rec, y_val_rec = create_sliding_window(val_scaled, window_size, horizon_r, 1)
X_test_rec, y_test_rec = create_sliding_window(test_scaled, window_size, horizon_r, 1)

print("Recursive Data Shapes:")
print("Train X:", np.shape(X_train_rec), "| Train y:", np.shape(y_train_rec))
print("Val X:  ", np.shape(X_val_rec), "| Val y:  ", np.shape(y_val_rec))
print("Test X: ", np.shape(X_test_rec), "| Test y: ", np.shape(y_test_rec))

In [ ]:
# build and summary
gru_model_rec = build_gru_model(
    input_shape=(X_train_rec.shape[1], X_train_rec.shape[2]), 
    output_dim=y_train_rec.shape[1]
)
gru_model_rec.summary()

In [ ]:
# Train the model
history_gru_rec = train_model_wrapper(gru_model_rec, 'gru', X_train_rec, y_train_rec, X_val_rec, y_val_rec, epochs=61, optimized=True)

In [ ]:
plot_training_history(history_gru_rec, "GRU_Model_recursive")

In [ ]:
# Save the model
gru_model_rec.save(f"gru_recursive_h_{horizon_r}.keras")

In [ ]:
# Load the model
loaded_rec_model = tf.keras.models.load_model(
    f"gru_recursive_h_{horizon_r}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
TARGET_HORIZON = 5              
# ==========================================
print(f"=== Generating GRU Recursive Predictions for t+{TARGET_HORIZON} ===")

valid_length = len(X_test_rec) - TARGET_HORIZON + 1
steps_to_predict = min(X_STEPS_TO_PLOT, valid_length)

y_pred_rec_scaled = generate_recursive_predictions(loaded_rec_model, X_test_rec, TARGET_HORIZON, steps_to_predict)

_, y_test_aligned_scaled = create_sliding_window(test_scaled, window_size, TARGET_HORIZON, direct=1)
y_test_aligned_scaled = y_test_aligned_scaled[:valid_length]

y_pred_unscaled = scaler.inverse_transform(y_pred_rec_scaled)
y_actual_unscaled = scaler.inverse_transform(y_test_aligned_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(actual, predicted, Y_LABEL, X_STEPS_TO_PLOT, FEATURE_NAME, "recursive", TARGET_HORIZON)

### Aggregated

In [ ]:
# Data
horizon_a = 5

X_train_agg, y_train_agg = create_sliding_window(train_scaled, window_size, horizon_a, 0)
X_val_agg, y_val_agg = create_sliding_window(val_scaled, window_size, horizon_a, 0)
X_test_agg, y_test_agg = create_sliding_window(test_scaled, window_size, horizon_a, 0)

print("Aggregated Data Shapes:")
print("Train X:", np.shape(X_train_agg), "| Train y:", np.shape(y_train_agg))
print("Val X:  ", np.shape(X_val_agg), "| Val y:  ", np.shape(y_val_agg))
print("Test X: ", np.shape(X_test_agg), "| Test y: ", np.shape(y_test_agg))

In [ ]:
# build and summary
gru_model_agg = build_gru_model(
    input_shape=(X_train_agg.shape[1], X_train_agg.shape[2]), 
    output_dim=y_train_agg.shape[1]
)
gru_model_agg.summary()

In [ ]:
# Train the model
history_gru_agg = train_model_wrapper(gru_model_agg, 'gru', X_train_agg, y_train_agg, X_val_agg, y_val_agg, epochs=61, optimized=True)

In [ ]:
plot_training_history(history_gru_agg, "GRU_Model_aggregated")

In [ ]:
# Save the model
gru_model_agg.save(f"gru_aggregated_h_{horizon_a}.keras")

In [ ]:
# Load the model

loaded_agg_model = tf.keras.models.load_model(
    f"gru_aggregated_h_{horizon_a}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
horizon_a = 5                   
# ==========================================

print(f"=== Generating GRU Aggregated Prediction for Average of t+1 to t+{horizon_a} ===")

y_pred_scaled = loaded_agg_model.predict(X_test_agg, verbose=0)

y_actual_unscaled = scaler.inverse_transform(y_test_agg) 
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="aggregated", 
    horizon=horizon_a
)

## Bi-LSTM

In [ ]:
def build_bi_lstm_model(input_shape, output_dim):
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # First Bi-LSTM layer (25 units)
    model.add(Bidirectional(LSTM(25, return_sequences=True)))

    # Second Bi-LSTM layer (25 units)
    model.add(Bidirectional(LSTM(25, return_sequences=False)))

    # Output layer
    model.add(Dense(output_dim, activation='linear'))
    
    model.compile(
        optimizer='adam', 
        loss='mse', 
        metrics=[
            'mae', 
            RootMeanSquaredError(name='rmse'), 
            MeanAbsolutePercentageError(name='mape'),
            tolerance_accuracy
        ]
    )
    
    return model

### Direct

In [ ]:
# Data Configuration
horizon_d = 1

X_train_dir, y_train_dir = create_sliding_window(train_scaled, window_size, horizon_d, 1)
X_val_dir, y_val_dir = create_sliding_window(val_scaled, window_size, horizon_d, 1)
X_test_dir, y_test_dir = create_sliding_window(test_scaled, window_size, horizon_d, 1)

print(f"Direct Training Shapes: X={X_train_dir.shape}, y={y_train_dir.shape}")

In [ ]:
# Initialize and Build
bi_lstm_model_dir = build_bi_lstm_model(
    input_shape=(X_train_dir.shape[1], X_train_dir.shape[2]), 
    output_dim=y_train_dir.shape[1]
)

bi_lstm_model_dir.summary()

In [ ]:
# Train the model
history_bi_lstm_dir = train_model_wrapper(bi_lstm_model_dir, 'bi-lstm', X_train_dir, y_train_dir, X_val_dir, y_val_dir, epochs=56, optimized=True)

In [ ]:
plot_training_history(history_bi_lstm_dir, "Bi_LSTM_Model_direct")

In [ ]:
# Save the model
bi_lstm_model_dir.save(f"bi_lstm_direct_h_{horizon_d}.keras")
print(f"Model saved: bi_lstm_direct_h_{horizon_d}.keras")

In [ ]:
# Load the model
loaded_bi_dir_model = tf.keras.models.load_model(
    f"bi_lstm_direct_h_{horizon_d}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
# ==========================================

print("=== Generating Bi-LSTM Direct Prediction Plot ===")

y_pred_scaled = loaded_bi_dir_model.predict(X_test_dir, verbose=0)

y_test_unscaled = scaler.inverse_transform(y_test_dir)
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_test_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="direct", 
    horizon=horizon_d
)

### Recursive

In [ ]:
# Recursive Data Configuration
horizon_r = 1

X_train_rec, y_train_rec = create_sliding_window(train_scaled, window_size, horizon_r, 1)
X_val_rec, y_val_rec = create_sliding_window(val_scaled, window_size, horizon_r, 1)
X_test_rec, y_test_rec = create_sliding_window(test_scaled, window_size, horizon_r, 1)

print(f"Recursive Training Shapes: X={X_train_rec.shape}, y={y_train_rec.shape}")

In [ ]:
# Initialize and Build
bi_lstm_model_rec = build_bi_lstm_model(
    input_shape=(X_train_rec.shape[1], X_train_rec.shape[2]), 
    output_dim=y_train_rec.shape[1]
)

bi_lstm_model_rec.summary()

In [ ]:
# Train the model
history_bi_lstm_rec = train_model_wrapper(bi_lstm_model_rec, 'bi-lstm', X_train_rec, y_train_rec, X_val_rec, y_val_rec, epochs=56, optimized=True)

In [ ]:
plot_training_history(history_bi_lstm_rec, "Bi_LSTM_Model_recursive")

In [ ]:
# Save the model
bi_lstm_model_rec.save(f"bi_lstm_recursive_h_{horizon_r}.keras")

In [ ]:
# Load the model
loaded_bi_rec_model = tf.keras.models.load_model(
    f"bi_lstm_recursive_h_{horizon_r}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
TARGET_HORIZON = 5
# ==========================================

print(f"=== Generating Bi-LSTM Recursive Predictions for t+{TARGET_HORIZON} ===")

valid_length = len(X_test_rec) - TARGET_HORIZON + 1
steps_to_predict = min(X_STEPS_TO_PLOT, valid_length)

y_pred_rec_scaled = generate_recursive_predictions(
    loaded_bi_rec_model, 
    X_test_rec, 
    TARGET_HORIZON, 
    steps_to_predict
)

_, y_test_aligned_scaled = create_sliding_window(test_scaled, window_size, TARGET_HORIZON, direct=1)
y_test_aligned_scaled = y_test_aligned_scaled[:valid_length]

y_pred_unscaled = scaler.inverse_transform(y_pred_rec_scaled)
y_actual_unscaled = scaler.inverse_transform(y_test_aligned_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(actual, predicted, Y_LABEL, X_STEPS_TO_PLOT, FEATURE_NAME, "recursive", TARGET_HORIZON)

### Aggregated

In [ ]:
# Aggregated Data Configuration
horizon_a = 5

X_train_agg, y_train_agg = create_sliding_window(train_scaled, window_size, horizon_a, 0)
X_val_agg, y_val_agg = create_sliding_window(val_scaled, window_size, horizon_a, 0)
X_test_agg, y_test_agg = create_sliding_window(test_scaled, window_size, horizon_a, 0)

print(f"Aggregated Training Shapes: X={X_train_agg.shape}, y={y_train_agg.shape}")

In [ ]:
# Initialize and Build
bi_lstm_model_agg = build_bi_lstm_model(
    input_shape=(X_train_agg.shape[1], X_train_agg.shape[2]), 
    output_dim=y_train_agg.shape[1]
)

bi_lstm_model_agg.summary()

In [ ]:
# Train the model
history_bi_lstm_agg = train_model_wrapper(bi_lstm_model_agg, 'bi-lstm', X_train_agg, y_train_agg, X_val_agg, y_val_agg, epochs=56, optimized=True)

In [ ]:
plot_training_history(history_bi_lstm_agg, "Bi_LSTM_Model_aggregated")

In [ ]:
# Save the model
bi_lstm_model_agg.save(f"bi_lstm_aggregated_h_{horizon_a}.keras")

In [ ]:
# Load the model
loaded_bi_agg_model = tf.keras.models.load_model(
    f"bi_lstm_aggregated_h_{horizon_a}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
# ==========================================

print(f"=== Generating Bi-LSTM Aggregated Prediction Plot (h={horizon_a}) ===")

y_pred_scaled = loaded_bi_agg_model.predict(X_test_agg, verbose=0)

y_actual_unscaled = scaler.inverse_transform(y_test_agg)
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="aggregated", 
    horizon=horizon_a
)

## CNN-LSTM

In [ ]:
def build_cnn_lstm_model(input_shape, output_dim):

    model = Sequential()
    
    # --- CNN ENCODER ---
    # Paper specifies kernel_size=3 for the hybrid model specifically
    model.add(Input(shape=input_shape))
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Flatten())
    
    # --- BRIDGE ---
    # RepeatVector connects the flattened CNN output to the 3D LSTM input
    model.add(RepeatVector(1))
    
    # --- LSTM DECODER ---
    # Two LSTM layers with 25 units each as per paper configuration
    model.add(LSTM(25, activation='relu', return_sequences=True))
    model.add(LSTM(25, activation='relu', return_sequences=False))
    
    # --- OUTPUT ---
    model.add(Dense(output_dim, activation='linear'))
    
    model.compile(
        optimizer='adam', 
        loss='mse', 
        metrics=[
            'mae', 
            RootMeanSquaredError(name='rmse'), 
            MeanAbsolutePercentageError(name='mape'),
            tolerance_accuracy
        ]
    )
    
    return model

### Direct

In [ ]:
# Data
horizon_d = 5

X_train_dir, y_train_dir = create_sliding_window(train_scaled, window_size, horizon_d, 1)
X_val_dir, y_val_dir = create_sliding_window(val_scaled, window_size, horizon_d, 1)
X_test_dir, y_test_dir = create_sliding_window(test_scaled, window_size, horizon_d, 1)

print(f"Direct Training Shapes: X={X_train_dir.shape}, y={y_train_dir.shape}")

In [ ]:
# Initialize and Build
cnn_lstm_model_dir = build_cnn_lstm_model(
    input_shape=(X_train_dir.shape[1], X_train_dir.shape[2]), 
    output_dim=y_train_dir.shape[1]
)

cnn_lstm_model_dir.summary()

In [ ]:
# Train the model
history_cnn_lstm_dir = train_model_wrapper(cnn_lstm_model_dir, 'cnn-lstm', X_train_dir, y_train_dir, X_val_dir, y_val_dir, epochs=24, optimized=True)

In [ ]:
plot_training_history(history_cnn_lstm_dir, "CNN_LSTM_Model_direct")

In [ ]:
# Save the model
cnn_lstm_model_dir.save(f"cnn_lstm_direct_h_{horizon_d}.keras")
print(f"Model successfully saved: cnn_lstm_direct_h_{horizon_d}.keras")

In [ ]:
# Load the model
loaded_cnn_lstm_dir = tf.keras.models.load_model(
    f"cnn_lstm_direct_h_{horizon_d}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
# ==========================================

print("=== Generating CNN-LSTM Direct Prediction Plot ===")

y_pred_scaled = loaded_cnn_lstm_dir.predict(X_test_dir, verbose=0)

y_test_unscaled = scaler.inverse_transform(y_test_dir)
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_test_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="direct", 
    horizon=horizon_d
)

### Recursive

In [ ]:
# Data
horizon_r = 1

X_train_rec, y_train_rec = create_sliding_window(train_scaled, window_size, horizon_r, 1)
X_val_rec, y_val_rec = create_sliding_window(val_scaled, window_size, horizon_r, 1)
X_test_rec, y_test_rec = create_sliding_window(test_scaled, window_size, horizon_r, 1)

print(f"Recursive Training Shapes: X={X_train_rec.shape}, y={y_train_rec.shape}")

In [ ]:
# Initialize and Build
cnn_lstm_model_rec = build_cnn_lstm_model(
    input_shape=(X_train_rec.shape[1], X_train_rec.shape[2]), 
    output_dim=y_train_rec.shape[1]
)

cnn_lstm_model_rec.summary()

In [ ]:
# Train the model
history_cnn_lstm_rec = train_model_wrapper(cnn_lstm_model_rec, 'cnn-lstm', X_train_rec, y_train_rec, X_val_rec, y_val_rec, epochs=24, optimized=True)

In [ ]:
plot_training_history(history_cnn_lstm_rec, "CNN_LSTM_Model_recursive")

In [ ]:
# Save the model
cnn_lstm_model_rec.save(f"cnn_lstm_recursive_h_{horizon_r}.keras")

In [ ]:
# Load the model
loaded_cnn_lstm_rec = tf.keras.models.load_model(
    f"cnn_lstm_recursive_h_{horizon_r}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
TARGET_HORIZON = 5              
# ==========================================

print(f"=== Generating CNN-LSTM Recursive Predictions for t+{TARGET_HORIZON} ===")

valid_length = len(X_test_rec) - TARGET_HORIZON + 1
steps_to_predict = min(X_STEPS_TO_PLOT, valid_length)

y_pred_rec_scaled = generate_recursive_predictions(
    loaded_cnn_lstm_rec, 
    X_test_rec, 
    TARGET_HORIZON, 
    steps_to_predict
)

_, y_test_aligned_scaled = create_sliding_window(test_scaled, window_size, TARGET_HORIZON, direct=1)
y_test_aligned_scaled = y_test_aligned_scaled[:valid_length]

y_pred_unscaled = scaler.inverse_transform(y_pred_rec_scaled)
y_actual_unscaled = scaler.inverse_transform(y_test_aligned_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(actual, predicted, Y_LABEL, X_STEPS_TO_PLOT, FEATURE_NAME, "recursive", TARGET_HORIZON)

### Aggregated

In [ ]:
# Data
horizon_a = 5

X_train_agg, y_train_agg = create_sliding_window(train_scaled, window_size, horizon_a, 0)
X_val_agg, y_val_agg = create_sliding_window(val_scaled, window_size, horizon_a, 0)
X_test_agg, y_test_agg = create_sliding_window(test_scaled, window_size, horizon_a, 0)

print(f"Aggregated Training Shapes: X={X_train_agg.shape}, y={y_train_agg.shape}")

In [ ]:
# Initialize and Build
cnn_lstm_model_agg = build_cnn_lstm_model(
    input_shape=(X_train_agg.shape[1], X_train_agg.shape[2]), 
    output_dim=y_train_agg.shape[1]
)

cnn_lstm_model_agg.summary()

In [ ]:
# Train the model
history_cnn_lstm_agg = train_model_wrapper(cnn_lstm_model_agg, 'cnn-lstm', X_train_agg, y_train_agg, X_val_agg, y_val_agg, epochs=24, optimized=True)

In [ ]:
plot_training_history(history_cnn_lstm_agg, "CNN_LSTM_Model_aggregated")

In [ ]:
# Save the model
cnn_lstm_model_agg.save(f"cnn_lstm_aggregated_h_{horizon_a}.keras")

In [ ]:
# Load the model
loaded_cnn_lstm_agg = tf.keras.models.load_model(
    f"cnn_lstm_aggregated_h_{horizon_a}.keras", 
    compile=False
)

In [ ]:
# ==========================================
FEATURE_INDEX = 0               
FEATURE_NAME = "UE1: web-rtc"   
Y_LABEL = "Throughput (Mbps)"   
X_STEPS_TO_PLOT = 150           
# ==========================================

print(f"=== Generating CNN-LSTM Aggregated Prediction Plot (h={horizon_a}) ===")

y_pred_scaled = loaded_cnn_lstm_agg.predict(X_test_agg, verbose=0)

y_actual_unscaled = scaler.inverse_transform(y_test_agg)
y_pred_unscaled = scaler.inverse_transform(y_pred_scaled)

actual = y_actual_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]
predicted = y_pred_unscaled[:X_STEPS_TO_PLOT, FEATURE_INDEX]

plot_actual_pred(
    actual=actual, 
    predicted=predicted, 
    y_label=Y_LABEL, 
    x_steps=X_STEPS_TO_PLOT, 
    feature_name=FEATURE_NAME, 
    model_type="aggregated", 
    horizon=horizon_a
)

# Heatmap

In [ ]:
def plot_filtered_r2_heatmap(models_dict, X_test, y_test, feature_names, filename="R2_Heatmap.pdf"):

    r2_results = {}
    
    # Calculate R2 for each model
    for model_name, model in models_dict.items():
        y_pred = model.predict(X_test, verbose=0)
        
        scores = r2_score(y_test, y_pred, multioutput='raw_values')
        
        scores = [max(0, score) for score in scores]
        r2_results[model_name] = scores

    df = pd.DataFrame(r2_results, index=feature_names).T
    
    # Filter out columns where all models scored exactly 0.0
    df_filtered = df.loc[:, (df != 0).any(axis=0)]

    # Plotting
    fig, ax = plt.subplots(figsize=(8, 4))
    
    sns.heatmap(
        df_filtered, 
        annot=True,
        fmt=".2f",
        cmap="RdYlGn",
        vmin=0.0, vmax=1.0,
        linewidths=0.5,
        cbar_kws={'label': 'R² Score'},
        ax=ax
    )
    
    ax.set_ylabel("Model Architecture", fontsize=10)
    
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    
    plt.tight_layout()
    plt.savefig(filename, format='pdf', bbox_inches='tight')
    plt.show()

In [ ]:
# direct models
my_direct_models = {
    "FNN": fnn_model_dir,
    "CNN": cnn_model_dir,
    "LSTM": lstm_model_dir,
    "GRU": gru_model_dir,
    "Bi-LSTM": bi_lstm_model_dir,
    "CNN-LSTM": cnn_lstm_model_dir
}


feature_names = [
    "UE1: web-rtc", "UE1: sipp", "UE1: web-server",
    "UE2: web-rtc", "UE2: sipp", "UE2: web-server",
    "UE3: web-rtc", "UE3: sipp", "UE3: web-server",
    "UE1-Jitter", "UE2-Jitter", "UE3-Jitter",
    "UE1-CQI", "UE2-CQI", "UE3-CQI"
]

print("Generating filtered R2 Heatmap...")
plot_filtered_r2_heatmap(
    models_dict=my_direct_models,
    X_test=X_test_dir,
    y_test=y_test_dir,
    feature_names=feature_names,
    filename="Direct_Models_R2_Heatmap.pdf"
)

In [ ]:
# aggregated models
my_aggregated_models = {
    "FNN": fnn_model_agg,
    "CNN": cnn_model_agg,
    "LSTM": lstm_model_agg,
    "GRU": gru_model_agg,
    "Bi-LSTM": bi_lstm_model_agg,
    "CNN-LSTM": cnn_lstm_model_agg
}

feature_names = [
    "UE1: web-rtc", "UE1: sipp", "UE1: web-server",
    "UE2: web-rtc", "UE2: sipp", "UE2: web-server",
    "UE3: web-rtc", "UE3: sipp", "UE3: web-server",
    "UE1-Jitter", "UE2-Jitter", "UE3-Jitter",
    "UE1-CQI", "UE2-CQI", "UE3-CQI"
]


print("Generating filtered R2 Heatmap for Aggregated Models...")
plot_filtered_r2_heatmap(
    models_dict=my_aggregated_models,
    X_test=X_test_agg,
    y_test=y_test_agg,
    feature_names=feature_names,
    filename="Aggregated_Models_R2_Heatmap.pdf"
)